# Watershed Peak SWE Year Selection

Ranks annual East River watershed peak iSnobal SWE and selects representative **Low**, **Typical**, **High**, and **Abnormal** water years.

- Low = smallest peak SWE among non-abnormal years
- Typical = peak SWE closest to the non-abnormal median
- High = largest peak SWE among non-abnormal years
- Abnormal = WY2026 by explicit project choice


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd


# ============================================================
# Settings
# ============================================================

OUTPUT_DIR = Path(
    "/bsuhome/zihaozhang/scratch/iSnobal/"
    "East_River_Workflow_Code_v0.8.2/output/output1"
)

SWE_FILE = OUTPUT_DIR / "tables" / "isnobal_watershed_daily.csv"

# Keep the scientifically unusual year separate from the
# low / typical / high peak-SWE ranking.
ABNORMAL_YEAR = 2026

# Output table
OUT_FILE = OUTPUT_DIR / "tables" / "watershed_peak_swe_year_ranking.csv"


# ============================================================
# Load watershed SWE
# ============================================================

df = pd.read_csv(SWE_FILE, parse_dates=["date"])

required = {"date", "water_year", "swe_mm"}
missing = required - set(df.columns)

if missing:
    raise KeyError(
        f"Missing required columns in {SWE_FILE}: {sorted(missing)}"
    )

df["swe_mm"] = pd.to_numeric(df["swe_mm"], errors="coerce")
df = df[np.isfinite(df["swe_mm"])].copy()
df["water_year"] = df["water_year"].astype(int)


# ============================================================
# Annual watershed peak SWE and peak date
# ============================================================

rows = []

for wy, sub in df.groupby("water_year"):
    sub = sub.sort_values("date")

    if sub.empty:
        continue

    idx = sub["swe_mm"].idxmax()

    rows.append(
        {
            "water_year": int(wy),
            "peak_SWE_mm": float(df.loc[idx, "swe_mm"]),
            "peak_date": df.loc[idx, "date"],
        }
    )

peak = pd.DataFrame(rows)

if peak.empty:
    raise RuntimeError("No valid annual peak SWE values were found.")


# ============================================================
# Rank all years from lowest to highest peak SWE
# ============================================================

peak = peak.sort_values("peak_SWE_mm").reset_index(drop=True)

peak["rank_low_to_high"] = np.arange(1, len(peak) + 1)

peak["percentile_rank"] = peak["peak_SWE_mm"].rank(
    method="average",
    pct=True,
)


# ============================================================
# Select abnormal year first
# ============================================================

if ABNORMAL_YEAR not in peak["water_year"].values:
    raise ValueError(
        f"ABNORMAL_YEAR={ABNORMAL_YEAR} is not present in the SWE table."
    )

peak["year_type"] = ""

peak.loc[
    peak["water_year"] == ABNORMAL_YEAR,
    "year_type"
] = "Abnormal"


# ============================================================
# Select LOW / TYPICAL / HIGH from all NON-abnormal years
# ============================================================

normal_pool = peak[
    peak["water_year"] != ABNORMAL_YEAR
].copy()

if len(normal_pool) < 3:
    raise RuntimeError(
        "Need at least three non-abnormal years to define "
        "low, typical, and high."
    )


# LOW = smallest annual peak SWE
low_idx = normal_pool["peak_SWE_mm"].idxmin()


# HIGH = largest annual peak SWE
high_idx = normal_pool["peak_SWE_mm"].idxmax()


# TYPICAL = annual peak SWE closest to median of non-abnormal years
median_peak = normal_pool["peak_SWE_mm"].median()

normal_pool["distance_from_median_mm"] = (
    normal_pool["peak_SWE_mm"] - median_peak
).abs()

typical_idx = normal_pool["distance_from_median_mm"].idxmin()


# Assign labels
peak.loc[low_idx, "year_type"] = "Low"
peak.loc[typical_idx, "year_type"] = "Typical"
peak.loc[high_idx, "year_type"] = "High"


# ============================================================
# Add anomaly relative to the non-abnormal median
# ============================================================

peak["difference_from_normal_median_mm"] = (
    peak["peak_SWE_mm"] - median_peak
)

peak["difference_from_normal_median_pct"] = (
    100.0
    * peak["difference_from_normal_median_mm"]
    / median_peak
)


# ============================================================
# Save full ranking
# ============================================================

peak.to_csv(OUT_FILE, index=False)


# ============================================================
# Print full ranking
# ============================================================

print("\nAnnual East River watershed peak SWE ranking")
print("Reference: iSnobal watershed SWE")
print("-" * 75)

print(
    peak[
        [
            "water_year",
            "peak_SWE_mm",
            "peak_date",
            "rank_low_to_high",
            "percentile_rank",
            "year_type",
        ]
    ].to_string(
        index=False,
        formatters={
            "peak_SWE_mm": "{:.1f}".format,
            "percentile_rank": "{:.3f}".format,
        },
    )
)


# ============================================================
# Print selected representative years
# ============================================================

selected = peak[
    peak["year_type"].isin(
        ["Low", "Typical", "High", "Abnormal"]
    )
].copy()

order = {
    "Low": 0,
    "Typical": 1,
    "High": 2,
    "Abnormal": 3,
}

selected["plot_order"] = selected["year_type"].map(order)
selected = selected.sort_values("plot_order")

print("\nSelected point-comparison years")
print("-" * 75)

print(
    selected[
        [
            "year_type",
            "water_year",
            "peak_SWE_mm",
            "peak_date",
            "difference_from_normal_median_mm",
            "difference_from_normal_median_pct",
        ]
    ].to_string(
        index=False,
        formatters={
            "peak_SWE_mm": "{:.1f}".format,
            "difference_from_normal_median_mm": "{:+.1f}".format,
            "difference_from_normal_median_pct": "{:+.1f}%".format,
        },
    )
)

print(f"\nMedian peak SWE of non-abnormal years = {median_peak:.1f} mm")
print(f"Saved ranking to:\n{OUT_FILE}")
